In [1]:
!pip install playwright nest-asyncio pandas playwright-stealth
!playwright install chromium firefox webkit

source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory
source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory


In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
import random
import time
import os
import re
from datetime import datetime, timedelta
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
from urllib.parse import quote

nest_asyncio.apply()

# ============================================================
# CONFIG
# ============================================================
# Google Sheet config — thay INPUT_FILE bằng link Google Sheet
# Cách lấy SHEET_ID: từ URL https://docs.google.com/spreadsheets/d/{SHEET_ID}/edit
GOOGLE_SHEET_ID = ""  # Để trống = đọc từ file local INPUT_FILE
GOOGLE_SHEET_NAME = "agoda"  # Tên sheet tab cần đọc (để trống = sheet đầu tiên)

INPUT_FILE = "./agoda2.csv"  # Fallback nếu không dùng Google Sheet
TEMP_OUTPUT_FILE = "TEMP_agoda2.csv"
OUTPUT_PREFIX = "FINAL_"

HEADLESS = True
DEBUG_SCREENSHOTS = False

CHECKIN_OFFSET = 2          # Số ngày cộng thêm kể từ ngày crawl (ví dụ: today + 3)
NUM_WORKERS = 6             # Anti-detect: chỉ 1 worker để giảm request rate
WEEKS_PER_HOTEL = 5        # Max weeks crawled in parallel per hotel
DAYS_PER_WEEK = 7           # Số ngày thử trong mỗi tuần (fallback từng ngày)
RETRIES_PER_DAY = 1         # Retry cho mỗi ngày (giảm vì đã thử nhiều ngày)
PAGE_TIMEOUT = 30000
BATCH_SIZE = 10
MAX_RETRY_ROUNDS = 1        # Giảm vì logic fallback đã cover
TARGET_NA_RATE = 0.10

AUTO_RETRY_NA_SOLDOUT = True  # Tự động crawl lại NA & SOLD OUT sau round 1

DELAY_RANGE = (3, 7)        # Anti-detect: delay dài hơn giữa các request
HOTEL_DELAY = (8, 15)       # Anti-detect: delay dài hơn giữa các hotel
RETRY_COOL_DOWN = (10, 20)  # Anti-detect: cool down dài hơn trước retry
RETRY_PAGE_TIMEOUT = [30000, 45000]

USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
]

# Anti-detect: Rotate browser profile — mỗi context giả lập 1 người dùng khác nhau
TIMEZONES = ["Asia/Ho_Chi_Minh", "Asia/Bangkok", "Asia/Singapore", "Asia/Tokyo", "Asia/Seoul"]
LOCALES = ["en-US", "en-GB", "vi-VN", "en-AU", "en-SG"]
SCREEN_RESOLUTIONS = [
    (1366, 768), (1440, 900), (1536, 864), (1600, 900),
    (1920, 1080), (1680, 1050), (1280, 800), (1792, 1120),
]

EXTRACT_PRICES_JS = """(targetRoom) => {
    const results = [];
    // 2026-03: Agoda changed from data-selenium to data-testid
    let masterRooms = document.querySelectorAll("[data-testid='room-item']");
    // Fallback to old selector for backward compatibility
    if (masterRooms.length === 0) {
        masterRooms = document.querySelectorAll("[data-selenium='MasterRoom']");
    }
    const targetLower = targetRoom.toLowerCase().trim();

    // Extract only the numeric price (with ₫ or VND), stripping all surrounding text
    // Handles Agoda variants:
    //   'Giá rẻ nhất từ trước đến nay!Nhanh lên! Phòng cuối cùng của chúng tôi!1.351.852₫Mỗi đêm, chưa có thuế'
    //   'Giá rẻ nhất từ trước đến nay!4 phòng cuối cùng của chúng tôi!4.590.370₫Mỗi đêm, chưa có thuế'
    //   'Giá rẻ nhất từ trước đến nay!-10%Số phòng có hạn900.000₫Mỗi đêm, chưa có thuế'
    //   'Số phòng có hạn2.383.333₫Mỗi đêm, chưa có thuế'
    function extractCleanPrice(text) {
        if (!text) return null;
        // Step 1: Strip known Agoda promotional/urgency prefixes & suffixes
        let cleaned = text;
        cleaned = cleaned.replace(/Giá rẻ nhất[^₫đ\d]*/gi, '');
        cleaned = cleaned.replace(/Nhanh lên[!.]?/gi, '');
        cleaned = cleaned.replace(/\d+\s*phòng cuối cùng[^₫đ\d]*/gi, '');
        cleaned = cleaned.replace(/Phòng cuối cùng[^₫đ\d]*/gi, '');
        cleaned = cleaned.replace(/Số phòng có hạn/gi, '');
        cleaned = cleaned.replace(/-?\d{1,3}%/g, '');
        cleaned = cleaned.replace(/Mỗi đêm[^₫đ]*/gi, '');
        cleaned = cleaned.replace(/chưa có thuế/gi, '');
        cleaned = cleaned.replace(/của chúng tôi[!.]?/gi, '');
        cleaned = cleaned.trim();

        // Step 2: Try to extract price from cleaned text first, then fallback to original
        function tryExtract(s) {
            const m1 = s.match(/(\d[\d.,]*\d)\s*₫/);
            if (m1) return m1[1].replace(/\s/g, '') + '₫';
            const m2 = s.match(/₫\s*(\d[\d.,]*\d)/);
            if (m2) return '₫' + m2[1].replace(/\s/g, '');
            const m3 = s.match(/(\d[\d.,]*\d)\s*VND/i);
            if (m3) return m3[1].replace(/\s/g, '') + ' VND';
            const m4 = s.match(/VND\s*(\d[\d.,]*\d)/i);
            if (m4) return 'VND ' + m4[1].replace(/\s/g, '');
            const m5 = s.match(/(\d[\d.,]{3,}\d)/);
            if (m5) return m5[1];
            return null;
        }
        return tryExtract(cleaned) || tryExtract(text);
    }

    function findPriceInText(text) {
        return extractCleanPrice(text);
    }

    const bodyText = document.body.innerText || '';
    const hotelSoldOut = /sold\s*out[!.]?\s*(our last room|all rooms)/i.test(bodyText);
    const noAvailability = /no\s*(rooms?)?\s*avail/i.test(bodyText) && masterRooms.length === 0;

    if ((hotelSoldOut || noAvailability) && masterRooms.length === 0) {
        return {found: false, soldOut: true, soldOutType: 'hotel', allRooms: 0, pageTitle: document.title, bodySnippet: bodyText.substring(0, 300)};
    }

    masterRooms.forEach(room => {
        // 2026-03: Try new selector first, fallback to old
        let nameEl = room.querySelector("[data-testid='room-name']");
        if (!nameEl) nameEl = room.querySelector("[data-selenium='masterroom-title-name']");
        const name = nameEl ? nameEl.textContent.trim() : '';
        const roomText = room.innerText || '';

        // Check for sold-out via data-testid
        let soldOutPrice = null;
        const soldOutEl = room.querySelector("[data-testid='sold-out-urgency']");
        if (soldOutEl) {
            const soPrice = extractCleanPrice(soldOutEl.textContent);
            if (soPrice) soldOutPrice = soPrice;
        }
        // Fallback: regex match
        if (!soldOutPrice) {
            const soldOutMatch = roomText.match(/sold\s*out\s*at\s*([₫đ]\s*[\d,. ]+|[\d,.]+\s*[₫đ])/i);
            if (soldOutMatch) soldOutPrice = soldOutMatch[1].trim();
        }
        if (!soldOutPrice) {
            let parent = room.parentElement;
            for (let i = 0; i < 3 && parent; i++) {
                const parentMatch = (parent.innerText || '').match(/sold\s*out\s*at\s*([₫đ]\s*[\d,. ]+|[\d,.]+\s*[₫đ])/i);
                if (parentMatch) { soldOutPrice = parentMatch[1].trim(); break; }
                parent = parent.parentElement;
            }
        }

        let prices = [];\n
\n
        // 2026-04: Helper to detect strikethrough/original price elements\n
        // Agoda shows: ~~original price~~ → discounted price → final price after cashback\n
        // We must SKIP the strikethrough price and take the FINAL price\n
        function isStrikethrough(el) {\n
            let node = el;\n
            for (let i = 0; i < 5 && node && node !== room; i++) {\n
                const style = window.getComputedStyle(node);\n
                if (style.textDecoration.includes('line-through') ||\n
                    style.textDecorationLine.includes('line-through')) return true;\n
                if (node.tagName === 'DEL' || node.tagName === 'S') return true;\n
                node = node.parentElement;\n
            }\n
            return false;\n
        }\n
\n
        // 2026-04: New strategy — collect ALL non-strikethrough prices, take the LAST one\n
        // On Agoda: last price = \"Price after Cashback\" = the actual price\n
        room.querySelectorAll(\"[data-testid='offer-price']\").forEach(p => {\n
            let offerPrices = [];\n
            const children = p.querySelectorAll('span, div, strong, b');\n
            for (const child of children) {\n
                // SKIP strikethrough (original/gach ngang) prices\n
                if (isStrikethrough(child)) continue;\n
                const ct = child.textContent.trim();\n
                // Look for elements that contain ONLY a price pattern\n
                if (/^[₫đ]?\\s*\\d[\\d.,\\s]*\\d\\s*[₫đ]?$/.test(ct)) {\n
                    const clean = extractCleanPrice(ct);\n
                    if (clean) offerPrices.push(clean);\n
                }\n
            }\n
            // Take the LAST non-strikethrough price (= final price after cashback)\n
            if (offerPrices.length > 0) {\n
                prices.push(offerPrices[offerPrices.length - 1]);\n
                return;\n
            }\n
\n
            // Fallback: extract from full text using regex\n
            const text = p.textContent.trim();\n
            if (!text) return;\n
            const clean = extractCleanPrice(text);\n
            if (clean) { prices.push(clean); return; }\n
            // Last resort: push raw text (should rarely happen now)\n
            if (text) prices.push(text);\n
        });\n
        // Fallback: old selectors
        if (prices.length === 0) {
            room.querySelectorAll("[data-selenium='PriceDisplay']").forEach(p => {
                const text = p.textContent.trim();
                if (text) {
                    const clean = extractCleanPrice(text);
                    prices.push(clean || text);
                }
            });
        }
        // Fallback: look for price in room-offer-price-info
        if (prices.length === 0) {
            room.querySelectorAll("[data-testid='room-offer-price-info']").forEach(p => {
                const text = p.textContent.trim();
                if (!text) return;
                const clean = extractCleanPrice(text);
                if (clean) { prices.push(clean); return; }
            });
        }
        // Fallback: text walker
        if (prices.length === 0) {
            const walker = document.createTreeWalker(room, NodeFilter.SHOW_TEXT);
            while (walker.nextNode()) {
                const price = extractCleanPrice(walker.currentNode.textContent.trim());
                if (price) prices.push(price);
            }
        }

        results.push({
            name, nameLower: name.toLowerCase().trim(),
            prices: prices.slice(0, 5),
            matched: name.toLowerCase().trim() === targetLower,
            soldOutPrice
        });
    });

    const target = results.find(r => r.matched);
    if (target) {
        if (target.prices.length > 0) return {found: true, price: target.prices[0], room: target.name, allRooms: results.length};
        if (target.soldOutPrice) return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: target.soldOutPrice, room: target.name, allRooms: results.length};
    }
    const partial = results.find(r => r.nameLower.includes(targetLower) || targetLower.includes(r.nameLower));
    if (partial) {
        if (partial.prices.length > 0) return {found: true, price: partial.prices[0], room: partial.name, allRooms: results.length, partial: true};
        if (partial.soldOutPrice) return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: partial.soldOutPrice, room: partial.name, allRooms: results.length, partial: true};
    }
    const allSoldOut = results.length > 0 && results.every(r => r.soldOutPrice && r.prices.length === 0);
    if (allSoldOut) {
        const rel = target || partial || results[0];
        return {found: false, soldOut: true, soldOutType: 'all_rooms', soldOutPrice: rel.soldOutPrice, allRooms: results.length};
    }
    return {found: false, soldOut: false, allRooms: results.length, roomNames: results.map(r => r.name), pageTitle: document.title, bodySnippet: (document.body.innerText || '').substring(0, 300)};
}"""

# ============================================================
# HELPERS
# ============================================================
def clean_price(price_str):
    """Clean price string: extract only the numeric price with currency symbol.
    Removes Agoda promotional/urgency text before extracting price.
    Handles variants:
        'Giá rẻ nhất từ trước đến nay!Nhanh lên! Phòng cuối cùng của chúng tôi!1.351.852₫Mỗi đêm, chưa có thuế' → '1.351.852₫'
        'Giá rẻ nhất từ trước đến nay!4 phòng cuối cùng của chúng tôi!4.590.370₫Mỗi đêm, chưa có thuế' → '4.590.370₫'
        'Giá rẻ nhất từ trước đến nay!-10%Số phòng có hạn900.000₫Mỗi đêm, chưa có thuế' → '900.000₫'
        'Số phòng có hạn2.383.333₫Mỗi đêm, chưa có thuế' → '2.383.333₫'
        '2.001.237₫Mỗi đêm, chưa có thuế' → '2.001.237₫'
        '₫ 4,305,919' → '₫ 4,305,919'
        'NA' → 'NA'
        'SOLD OUT' → 'SOLD OUT'
    """
    if not price_str or price_str in ('NA', 'nan', ''):
        return 'NA'
    s = str(price_str).strip()
    if s.startswith('SOLD OUT'):
        return s
    # Step 1: Strip known Agoda promotional/urgency prefixes & suffixes
    cleaned = s
    cleaned = re.sub(r'Giá rẻ nhất[^₫đ\d]*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'Nhanh lên[!.]?', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\d+\s*phòng cuối cùng[^₫đ\d]*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'Phòng cuối cùng[^₫đ\d]*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'Số phòng có hạn', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'-?\d{1,3}%', '', cleaned)
    cleaned = re.sub(r'Mỗi đêm[^₫đ]*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'chưa có thuế', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'của chúng tôi[!.]?', '', cleaned, flags=re.IGNORECASE)
    cleaned = cleaned.strip()

    # Step 2: Try extract from cleaned text first, then fallback to original
    def try_extract(txt):
        m = re.search(r'(\d[\d.,]*\d)\s*₫', txt)
        if m: return m.group(1) + '₫'
        m = re.search(r'₫\s*(\d[\d.,]*\d)', txt)
        if m: return '₫ ' + m.group(1)
        m = re.search(r'(\d[\d.,]*\d)\s*VND', txt, re.IGNORECASE)
        if m: return m.group(1) + ' VND'
        m = re.search(r'VND\s*(\d[\d.,]*\d)', txt, re.IGNORECASE)
        if m: return 'VND ' + m.group(1)
        if re.match(r'^[₫đ]?\s*[\d.,\s]+[₫đ]?$', txt): return txt
        m = re.search(r'(\d[\d.,]{3,}\d)', txt)
        if m: return m.group(1)
        return None

    result = try_extract(cleaned) or try_extract(s)
    return result if result else s

def read_hotels_from_source():
    """Đọc hotel list từ Google Sheet (nếu có SHEET_ID) hoặc file local."""
    if GOOGLE_SHEET_ID:
        return read_hotels_from_gsheet(GOOGLE_SHEET_ID, GOOGLE_SHEET_NAME)
    else:
        return read_hotels_from_csv(INPUT_FILE)

def read_hotels_from_gsheet(sheet_id, sheet_name=""):
    """Đọc data từ Google Sheet published dưới dạng CSV."""
    try:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv"
        if sheet_name:
            url += f"&sheet={quote(sheet_name)}"
        print(f"📡 Đọc Google Sheet: ...{sheet_id[-8:]} | tab: {sheet_name or '(default)'}", flush=True)
        df = pd.read_csv(url)
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        if not all(col in df.columns for col in required_cols):
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        print(f"✅ {len(df)} hotels từ Google Sheet", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"❌ Lỗi đọc Google Sheet: {e}", flush=True)
        print(f"💡 Kiểm tra: Sheet đã publish chưa? (File → Share → Publish to web)", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        if not all(col in df.columns for col in required_cols):
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        print(f"✅ {len(df)} hotels từ {file_path}", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"❌ Lỗi đọc CSV: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def save_backup_csv(all_week_prices, filename):
    """Save CSV — only-improve (không ghi NA lên giá thật) + atomic write + hotel mới cuối file."""
    try:
        rows = []
        written_keys = set()

        # Đọc file cũ để giữ thứ tự + không mất hotel + không ghi đè giá tốt
        if os.path.exists(filename):
            try:
                df_old = pd.read_csv(filename, keep_default_na=False, na_values=[])
                for _, row in df_old.iterrows():
                    k = (str(row.get("hotel_name", "")), str(row.get("room_type", "")))
                    written_keys.add(k)
                    if k in all_week_prices:
                        new_row = {"hotel_name": k[0], "room_type": k[1]}
                        for i in range(1, 7):
                            old_val = str(row.get(f"price_w{i}", "NA")).strip()
                            new_val = str(all_week_prices[k].get(f"Price W{i}", "NA")).strip()
                            # Chỉ cập nhật nếu giá mới TỐT HƠN — không ghi NA lên giá thật
                            if new_val in ("NA", "nan", "") and old_val not in ("NA", "nan", ""):
                                new_row[f"price_w{i}"] = old_val
                            else:
                                new_row[f"price_w{i}"] = new_val if new_val not in ("nan", "") else "NA"
                        rows.append(new_row)
                    else:
                        rows.append(row.to_dict())
            except:
                pass

        # Hotel mới → thêm cuối file
        for (hotel, room), prices in all_week_prices.items():
            if (hotel, room) not in written_keys:
                row = {"hotel_name": hotel, "room_type": room}
                for i in range(1, 7):
                    row[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
                rows.append(row)

        # Atomic write: ghi file tạm rồi rename → user không bao giờ thấy file rỗng
        tmp = filename + ".tmp"
        pd.DataFrame(rows).to_csv(tmp, index=False)
        os.replace(tmp, filename)
    except Exception as e:
        print(f"❌ Error saving: {e}", flush=True)

def update_url_checkin(url, checkin_date):
    new_date = checkin_date.strftime("%Y-%m-%d")
    if 'checkin=' in url.lower():
        return re.sub(r'checkin=[\d-]+', f'checkin={new_date}', url, flags=re.IGNORECASE)
    return f"{url}{'&' if '?' in url else '?'}checkin={new_date}"

def calc_na_stats(d):
    return sum(1 for i in range(1, 7) if d.get(f"Price W{i}", "NA") == "NA"), 6

def calc_batch_na_rate(awp, keys):
    tc, nc = 0, 0
    for k in keys:
        if k in awp:
            n, t = calc_na_stats(awp[k])
            nc += n; tc += t
    return nc / max(tc, 1), nc, tc

def find_retry_weeks(awp, keys):
    items = []
    for k in keys:
        if k not in awp: continue
        p = awp[k]
        has_real = any(v != "NA" and not str(v).startswith("SOLD OUT") for v in p.values())
        for i in range(1, 7):
            v = p.get(f"Price W{i}", "NA")
            if v == "NA" or (str(v).startswith("SOLD OUT") and not has_real):
                items.append((k, i))
    return items

def find_na_soldout_weeks(awp, keys):
    """Tìm tất cả weeks có NA hoặc SOLD OUT để auto-retry."""
    items = []
    for k in keys:
        if k not in awp: continue
        p = awp[k]
        for i in range(1, 7):
            v = p.get(f"Price W{i}", "NA")
            if v == "NA" or str(v).startswith("SOLD OUT"):
                items.append((k, i))
    return items

# ============================================================
# HUMAN-LIKE BEHAVIOR — giả lập hành vi người thật
# ============================================================
async def human_like_browse(page):
    """Giả lập hành vi duyệt web tự nhiên: scroll, hover, di chuột."""
    # Scroll chậm xuống từng đoạn (giống người đọc)
    scroll_height = await page.evaluate("document.body.scrollHeight")
    current = 0
    while current < scroll_height * 0.6:
        step = random.randint(150, 400)
        current += step
        await page.evaluate(f"window.scrollTo({{top: {current}, behavior: 'smooth'}})")
        await asyncio.sleep(random.uniform(0.3, 0.8))

    # Random hover vào 1-2 element trên page
    try:
        links = page.locator("a[href]")
        count = await links.count()
        if count > 3:
            idx = random.randint(0, min(count - 1, 10))
            await links.nth(idx).hover(timeout=2000)
            await asyncio.sleep(random.uniform(0.2, 0.5))
    except:
        pass

    # Di chuột random
    try:
        await page.mouse.move(
            random.randint(100, 800),
            random.randint(100, 500)
        )
        await asyncio.sleep(random.uniform(0.1, 0.3))
    except:
        pass

# ============================================================
# CRAWL 1 NGÀY — thử 1 checkin date cụ thể
# ============================================================
async def crawl_single_day(browser, hotel_url, room_type, week_num, checkin,
                           retries=None, page_timeout=None, hotel_name=""):
    if retries is None: retries = RETRIES_PER_DAY
    if page_timeout is None: page_timeout = PAGE_TIMEOUT

    result = {"week": week_num, "price": "NA", "date": checkin.strftime('%Y-%m-%d')}

    stealth = Stealth()

    for retry in range(retries):
        context = None
        try:
            if retry > 0:
                backoff = random.uniform(5, 10) * (retry + 1)
                await asyncio.sleep(backoff)

            # Anti-detect: rotate browser profile mỗi request
            res = random.choice(SCREEN_RESOLUTIONS)
            context = await browser.new_context(
                viewport={"width": res[0], "height": res[1]},
                user_agent=random.choice(USER_AGENTS),
                locale=random.choice(LOCALES),
                timezone_id=random.choice(TIMEZONES),
                color_scheme=random.choice(["light", "dark", "no-preference"]),
                device_scale_factor=random.choice([1, 1.25, 1.5, 2]),
                java_script_enabled=True,
                has_touch=random.choice([True, False]),
            )
            page = await context.new_page()

            # Anti-detect: dùng playwright-stealth thay vì script thủ công
            await stealth.apply_stealth_async(page)

            url = update_url_checkin(hotel_url, checkin)

            try:
                await page.goto(url, timeout=page_timeout, wait_until="domcontentloaded")
            except Exception:
                pass

            await asyncio.sleep(random.uniform(2, 5))

            # Anti-detect: human-like browsing behavior
            await human_like_browse(page)

            try:
                close_btn = page.locator(".ab-close-button")
                if await close_btn.count() > 0:
                    await close_btn.first.click(timeout=2000)
            except: pass

            try:
                await page.wait_for_selector("[data-testid='offer-price'], [data-selenium='PriceDisplay']", timeout=15000)
            except:
                try:
                    await page.wait_for_selector("#property-room-grid-root, [data-testid='room-grid-element'], div#roomGrid", timeout=8000)
                    await asyncio.sleep(3)
                except:
                    await asyncio.sleep(3)

            await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2)")
            await asyncio.sleep(random.uniform(0.5, 1.5))
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(random.uniform(1.0, 2.5))

            extraction = await page.evaluate(EXTRACT_PRICES_JS, room_type)

            if extraction.get('found'):
                price = clean_price(extraction['price'])
                result["price"] = price
                return result

            if extraction.get('soldOut'):
                st = extraction.get('soldOutType', '')
                sp = extraction.get('soldOutPrice', '')
                price = "SOLD OUT" if st == 'hotel' else f"SOLD OUT {sp}".strip()
                result["price"] = price
                return result

            if retry == retries - 1:
                rooms = extraction.get('allRooms', 0)
                names = extraction.get('roomNames', [])[:3]
                result["debug_rooms"] = rooms
                result["debug_names"] = names

        except Exception as e:
            err = str(e)
            if "has been closed" in err or "Target page" in err:
                pass
        finally:
            if context:
                try: await context.close()
                except: pass

        await asyncio.sleep(random.uniform(*DELAY_RANGE))

    return result

# ============================================================
# CRAWL 1 TUẦN — thử từng ngày trong tuần cho đến khi có giá
# W1: base → base+6, W2: base+7 → base+13, ...
# ============================================================
async def crawl_week_range(browser, hotel_url, room_type, week_num, base_checkin,
                           hotel_name="", days_in_week=None, page_timeout=None):
    if days_in_week is None: days_in_week = DAYS_PER_WEEK

    week_start_offset = (week_num - 1) * 7
    week_start = base_checkin + timedelta(days=week_start_offset)
    week_end = week_start + timedelta(days=days_in_week - 1)

    print(f"      🔍 [{hotel_name[:20]}] W{week_num}: trying {week_start.strftime('%m/%d')}→{week_end.strftime('%m/%d')} ...", flush=True)

    last_result = {"week": week_num, "price": "NA", "date": ""}
    has_sold_out = False
    sold_out_count = 0
    na_count = 0

    for day_offset in range(days_in_week):
        checkin = base_checkin + timedelta(days=week_start_offset + day_offset)
        result = await crawl_single_day(
            browser, hotel_url, room_type, week_num, checkin,
            page_timeout=page_timeout, hotel_name=hotel_name
        )

        price = result["price"]

        # Tìm được giá → trả về ngay
        if price != "NA" and not str(price).startswith("SOLD OUT"):
            print(f"      ✅ [{hotel_name[:20]}] W{week_num}: {price} | {checkin.strftime('%Y-%m-%d')} (day {day_offset+1}/{days_in_week})", flush=True)
            return result

        if str(price).startswith("SOLD OUT"):
            has_sold_out = True
            sold_out_count += 1
            last_result = result
        else:
            na_count += 1
            last_result = result

        # Delay giữa các ngày trong tuần
        if day_offset < days_in_week - 1:
            await asyncio.sleep(random.uniform(1.5, 3.5))

    # Hết tất cả ngày trong tuần → quyết định kết quả
    if has_sold_out:
        last_result["price"] = "SOLD OUT"
        print(f"      🚫 [{hotel_name[:20]}] W{week_num}: SOLD OUT (tried {days_in_week} days: {sold_out_count} sold, {na_count} NA)", flush=True)
    else:
        print(f"      ❌ [{hotel_name[:20]}] W{week_num}: NA after trying all {days_in_week} days", flush=True)

    return last_result

# ============================================================
# PROCESS 1 HOTEL — lưu temp ngay sau mỗi week crawl xong
# ============================================================
async def process_hotel(browser, hotel_info, prev_data, base_checkin, semaphore, awp=None):
    hotel_name, hotel_url, room_type = hotel_info
    key = (hotel_name, room_type)

    if key in prev_data:
        vals = [prev_data[key].get(f"Price W{i}", "NA") for i in range(1, 7)]
        all_real = all(v != "NA" and not str(v).startswith("SOLD OUT") for v in vals)
        if all_real:
            return key, prev_data[key], True

    async with semaphore:
        await asyncio.sleep(random.uniform(*HOTEL_DELAY))
        print(f"\n🏨 {hotel_name} | {room_type}", flush=True)

        prices = {}
        weeks_to_crawl = []
        for wn in range(1, 7):
            kp = f"Price W{wn}"
            cached = prev_data[key].get(kp, "NA") if key in prev_data else "NA"
            if cached != "NA" and not str(cached).startswith("SOLD OUT"):
                prices[kp] = cached  # Giá thật → giữ, không crawl lại
            else:
                weeks_to_crawl.append(wn)  # NA hoặc SOLD OUT → crawl lại

        # Khởi tạo awp[key] với cached values
        if awp is not None:
            if key not in awp:
                awp[key] = {f"Price W{i}": "NA" for i in range(1, 7)}
            for kp, v in prices.items():
                awp[key][kp] = v

        if weeks_to_crawl:
            # Crawl tuần tự W1→W6 (giống người thật + dễ đọc log)
            for wn in weeks_to_crawl:
                r = await crawl_week_range(
                    browser, hotel_url, room_type, wn, base_checkin,
                    hotel_name=hotel_name
                )
                prices[f"Price W{r['week']}"] = r["price"]
                # Lưu ngay vào temp sau mỗi week
                if awp is not None:
                    awp[key][f"Price W{r['week']}"] = r["price"]
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                    print(f"      💾 [{hotel_name[:20]}] Saved W{r['week']} to temp", flush=True)

        na_c, _ = calc_na_stats(prices)
        so_c = sum(1 for i in range(1, 7) if str(prices.get(f"Price W{i}", "")).startswith("SOLD OUT"))
        icon = "✅" if na_c == 0 and so_c == 0 else f"⚠️({na_c}NA)" if na_c else f"🚫({so_c}SO)"
        print(f"   {icon} DONE: {hotel_name}", flush=True)
        return key, prices, False

# ============================================================
# RETRY — cũng dùng crawl_week_range
# ============================================================
async def retry_batch(browser, batch_infos, batch_keys, awp, base_checkin):
    ki = {(i[0], i[2]): i for i in batch_infos}
    for rn in range(1, MAX_RETRY_ROUNDS + 1):
        nr, nc, tc = calc_batch_na_rate(awp, batch_keys)
        if nr <= TARGET_NA_RATE: break
        items = find_retry_weeks(awp, batch_keys)
        if not items: break

        to = RETRY_PAGE_TIMEOUT[min(rn-1, len(RETRY_PAGE_TIMEOUT)-1)]
        print(f"\n🔁 RETRY {rn}/{MAX_RETRY_ROUNDS} | {len(items)} cells", flush=True)
        await asyncio.sleep(random.uniform(*RETRY_COOL_DOWN))

        hna = {}
        for k, wn in items:
            hna.setdefault(k, []).append(wn)

        sem = asyncio.Semaphore(NUM_WORKERS)
        async def do_retry(k, weeks):
            async with sem:
                info = ki.get(k)
                if not info: return
                hn, hu, rt = info
                await asyncio.sleep(random.uniform(*HOTEL_DELAY))
                wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
                async def rw(wn):
                    async with wsem:
                        return await crawl_week_range(
                            browser, hu, rt, wn, base_checkin,
                            hotel_name=hn, page_timeout=to
                        )
                results = await asyncio.gather(*[rw(w) for w in weeks])
                for r in results:
                    if r["price"] != "NA":
                        awp[k][f"Price W{r['week']}"] = r["price"]
        await asyncio.gather(*[do_retry(k, w) for k, w in hna.items()])

# ============================================================
# AUTO RETRY NA & SOLD OUT — Round 2 sau khi crawl xong tất cả
# ============================================================
async def auto_retry_na_soldout(browser, all_infos, awp, base_checkin):
    """Tự động crawl lại tất cả weeks có NA hoặc SOLD OUT sau round 1."""
    ki = {(i[0], i[2]): i for i in all_infos}
    all_keys = list(awp.keys())

    items = find_na_soldout_weeks(awp, all_keys)
    if not items:
        print(f"\n✅ Không có NA/SOLD OUT nào cần retry!", flush=True)
        return 0

    hna = {}
    for k, wn in items:
        hna.setdefault(k, []).append(wn)

    na_count = sum(1 for k, wn in items if awp[k].get(f"Price W{wn}", "NA") == "NA")
    so_count = sum(1 for k, wn in items if str(awp[k].get(f"Price W{wn}", "")).startswith("SOLD OUT"))

    print(f"\n{'='*60}", flush=True)
    print(f"🔄 AUTO ROUND 2 — Re-crawl NA & SOLD OUT", flush=True)
    print(f"   📊 {len(items)} cells ({na_count} NA + {so_count} SOLD OUT) across {len(hna)} hotels", flush=True)
    print(f"{'='*60}", flush=True)

    updated = 0
    sem = asyncio.Semaphore(NUM_WORKERS)

    async def do_hotel_retry(k, weeks):
        nonlocal updated
        async with sem:
            info = ki.get(k)
            if not info: return
            hn, hu, rt = info

            await asyncio.sleep(random.uniform(*HOTEL_DELAY))
            print(f"\n   🏨 [R2] {hn} | weeks: {weeks}", flush=True)

            wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
            async def rw(wn):
                async with wsem:
                    return await crawl_week_range(
                        browser, hu, rt, wn, base_checkin,
                        hotel_name=hn, page_timeout=45000
                    )
            results = await asyncio.gather(*[rw(w) for w in weeks])
            for r in results:
                new_price = r["price"]
                old_price = awp[k].get(f"Price W{r['week']}", "NA")
                if new_price != old_price:
                    if new_price != "NA":
                        awp[k][f"Price W{r['week']}"] = new_price
                        old_label = "NA" if old_price == "NA" else "SO"
                        new_label = new_price if not str(new_price).startswith("SOLD OUT") else "SOLD OUT"
                        print(f"      🔄 W{r['week']}: {old_label} → {new_label}", flush=True)
                        updated += 1

    await asyncio.gather(*[do_hotel_retry(k, w) for k, w in hna.items()])

    print(f"\n{'─'*50}", flush=True)
    print(f"   🔄 Round 2 complete: {updated}/{len(items)} cells updated", flush=True)
    print(f"{'─'*50}", flush=True)

    return updated

# ============================================================
# MAIN
# ============================================================
async def main():
    t0 = time.time()

    df = read_hotels_from_source()
    if len(df) == 0: return

    awp, prev = {}, {}

    def load_prev(fp):
        n = 0
        try:
            dp = pd.read_csv(fp, keep_default_na=False, na_values=[])
            for _, row in dp.iterrows():
                k = (row["hotel_name"], row["room_type"])
                if k in prev:
                    for i in range(1, 7):
                        v = str(row.get(f"price_w{i}", "NA")).strip()
                        if v and v not in ("NA", "nan"):
                            prev[k][f"Price W{i}"] = v
                else:
                    p = {}
                    for i in range(1, 7):
                        v = str(row.get(f"price_w{i}", "NA")).strip()
                        p[f"Price W{i}"] = "NA" if (not v or v in ("nan", "NA")) else v
                    prev[k] = p; awp[k] = p
                n += 1
        except: pass
        return n

    if os.path.exists(TEMP_OUTPUT_FILE):
        n = load_prev(TEMP_OUTPUT_FILE)
        print(f"📂 Loaded {n} hotels from temp", flush=True)

    # base_checkin = ngày mai (ngày đầu tiên của W1)
    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=CHECKIN_OFFSET)
    all_infos = [(r['hotel_name'], r['hotel_url'], r['room_type']) for _, r in df.iterrows()]

    # Phân loại: hotel mới (ưu tiên) → chưa đủ (NA/SOLD OUT) → đã đủ (skip)
    new_hotels = []
    incomplete_hotels = []
    complete_hotels = []
    for info in all_infos:
        key = (info[0], info[2])
        if key not in prev:
            new_hotels.append(info)
        else:
            vals = [prev[key].get(f"Price W{i}", "NA") for i in range(1, 7)]
            all_real = all(v != "NA" and not str(v).startswith("SOLD OUT") for v in vals)
            if all_real:
                awp[key] = prev[key]  # Đưa thẳng vào kết quả, không crawl
                complete_hotels.append(info)
            else:
                na_c = sum(1 for v in vals if v == "NA")
                so_c = sum(1 for v in vals if str(v).startswith("SOLD OUT"))
                incomplete_hotels.append((info, na_c, so_c))

    # Ưu tiên: hotel mới trước → incomplete sau
    infos = new_hotels + [item[0] for item in incomplete_hotels]
    total = len(infos)

    print(f"\n📊 Phân loại:", flush=True)
    print(f"   🆕 {len(new_hotels)} hotel mới (crawl trước)", flush=True)
    print(f"   🔄 {len(incomplete_hotels)} hotel chưa đủ (crawl NA/SOLD OUT)", flush=True)
    print(f"   ✅ {len(complete_hotels)} hotel đã đủ 6w (skip)", flush=True)
    if complete_hotels:
        for info in complete_hotels[:5]:
            print(f"      ⏭️ {info[0]}", flush=True)
        if len(complete_hotels) > 5:
            print(f"      ... và {len(complete_hotels) - 5} hotel khác", flush=True)

    # Show date ranges per week
    print(f"\n📅 Week date ranges (try each day until price found):", flush=True)
    for wn in range(1, 7):
        w_start = bc + timedelta(days=(wn - 1) * 7)
        w_end = w_start + timedelta(days=DAYS_PER_WEEK - 1)
        print(f"   W{wn}: {w_start.strftime('%Y-%m-%d (%a)')} → {w_end.strftime('%Y-%m-%d (%a)')}", flush=True)

    print(f"\n{'='*60}", flush=True)
    print(f"🎭 CRAWL v10 — anti-detect stealth mode", flush=True)
    print(f"📊 {total} hotels | {NUM_WORKERS}W × {WEEKS_PER_HOTEL}wk | {DAYS_PER_WEEK} days/week", flush=True)
    print(f"🛡️ Stealth: playwright-stealth + profile rotation + human behavior", flush=True)
    print(f"🔄 Auto Round 2: {'ON' if AUTO_RETRY_NA_SOLDOUT else 'OFF'}", flush=True)
    print(f"📡 Source: {'Google Sheet' if GOOGLE_SHEET_ID else 'Local file'}", flush=True)
    print(f"{'='*60}\n", flush=True)

    async with async_playwright() as p:
        if HEADLESS:
            browser = await p.chromium.launch(
                headless=False,
                args=[
                    '--headless=new',
                    '--disable-blink-features=AutomationControlled',
                    '--no-sandbox',
                    '--disable-dev-shm-usage',
                ],
            )
            print("✅ Browser launched (Chrome New Headless + Stealth)", flush=True)
        else:
            browser = await p.chromium.launch(
                headless=False,
                args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-dev-shm-usage'],
            )
            print("✅ Browser launched (visible window + Stealth)", flush=True)

        tb = (total + BATCH_SIZE - 1) // BATCH_SIZE
        ct = 0

        # ── ROUND 1: Crawl tất cả hotels ──
        for bi in range(tb):
            bs, be = bi * BATCH_SIZE, min((bi+1) * BATCH_SIZE, total)
            batch = infos[bs:be]

            print(f"\n{'='*60}", flush=True)
            print(f"📦 BATCH {bi+1}/{tb} | Hotels {bs+1}-{be}/{total}", flush=True)
            print(f"{'='*60}", flush=True)

            sem = asyncio.Semaphore(NUM_WORKERS)
            tasks = [process_hotel(browser, i, prev, bc, sem, awp=awp) for i in batch]

            bkeys = []
            for coro in asyncio.as_completed(tasks):
                try:
                    k, prices, skip = await coro
                    awp[k] = prices; bkeys.append(k)
                    if not skip: ct += 1
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                except Exception as e:
                    print(f"❌ {e}", flush=True)

            nr, nc, tc = calc_batch_na_rate(awp, bkeys)
            print(f"\n📊 Batch {bi+1}: NA = {nr:.1%} ({nc}/{tc})", flush=True)
            if nr > TARGET_NA_RATE:
                await retry_batch(browser, batch, bkeys, awp, bc)
                save_backup_csv(awp, TEMP_OUTPUT_FILE)

            # Report
            print(f"\n{'─'*50}", flush=True)
            for k in bkeys:
                pr = awp[k]; parts = []
                for i in range(1, 7):
                    v = pr.get(f"Price W{i}", "NA")
                    parts.append("✓" if v != "NA" and not str(v).startswith("SOLD OUT") else "🚫" if str(v).startswith("SOLD OUT") else "✗")
                nac, _ = calc_na_stats(pr)
                print(f"   {'✅' if nac==0 else '⚠️'} {k[0][:35]:35s} {' '.join(parts)}", flush=True)
            print(f"{'─'*50} | ⏱️ {int((time.time()-t0)//60)}m | {ct}/{total}", flush=True)

        # ── Save file final sau Round 1 ──
        fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
        save_backup_csv(awp, fn)
        print(f"\n📁 Round 1 saved: {fn}", flush=True)

        # ── Round 1 stats ──
        tc_r1 = len(awp) * 6
        na_r1 = sum(1 for p in awp.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
        so_r1 = sum(1 for p in awp.values() for i in range(1,7) if str(p.get(f"Price W{i}","")).startswith("SOLD OUT"))
        print(f"   Round 1: ✅ {tc_r1-na_r1-so_r1}/{tc_r1} | 🚫 {so_r1} SO | ❌ {na_r1} NA", flush=True)

        # ── ROUND 2: Auto retry NA & SOLD OUT ──
        if AUTO_RETRY_NA_SOLDOUT and (na_r1 > 0 or so_r1 > 0):
            r2_updated = await auto_retry_na_soldout(browser, infos, awp, bc)

            if r2_updated > 0:
                save_backup_csv(awp, fn)
                save_backup_csv(awp, TEMP_OUTPUT_FILE)
                print(f"\n📁 Final updated: {fn} ({r2_updated} cells changed)", flush=True)
            else:
                print(f"\n📁 Final unchanged (no improvements in Round 2)", flush=True)

        await browser.close()

    # ── Final stats ──
    tt = time.time() - t0; tc = len(awp) * 6
    na_t = sum(1 for p in awp.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
    so_t = sum(1 for p in awp.values() for i in range(1,7) if str(p.get(f"Price W{i}","")).startswith("SOLD OUT"))
    print(f"\n{'='*60}", flush=True)
    print(f"✅ FINAL COMPLETED | {fn}", flush=True)
    print(f"   ✅ Price: {tc-na_t-so_t}/{tc} ({(tc-na_t-so_t)/max(tc,1):.1%})", flush=True)
    print(f"   🚫 Sold:  {so_t}/{tc} | ❌ NA: {na_t}/{tc}", flush=True)
    print(f"⏱️ {int(tt//60)}m {int(tt%60)}s", flush=True)
    print(f"{'='*60}", flush=True)

await main()

✅ 20 hotels từ ./agoda2.csv
📂 Loaded 20 hotels from temp

📊 Phân loại:
   🆕 0 hotel mới (crawl trước)
   🔄 5 hotel chưa đủ (crawl NA/SOLD OUT)
   ✅ 15 hotel đã đủ 6w (skip)
      ⏭️ Harvest Day Hoi An - Hotel
      ⏭️ Little Oasis - Hotel
      ⏭️ Grand Sunrise Palace Hội An - Hotel
      ⏭️ Reu Boutique Hotel - Hotel
      ⏭️ Maison Vy - Hotel
      ... và 10 hotel khác

📅 Week date ranges (try each day until price found):
   W1: 2026-04-23 (Thu) → 2026-04-29 (Wed)
   W2: 2026-04-30 (Thu) → 2026-05-06 (Wed)
   W3: 2026-05-07 (Thu) → 2026-05-13 (Wed)
   W4: 2026-05-14 (Thu) → 2026-05-20 (Wed)
   W5: 2026-05-21 (Thu) → 2026-05-27 (Wed)
   W6: 2026-05-28 (Thu) → 2026-06-03 (Wed)

🎭 CRAWL v10 — anti-detect stealth mode
📊 5 hotels | 6W × 5wk | 7 days/week
🛡️ Stealth: playwright-stealth + profile rotation + human behavior
🔄 Auto Round 2: ON
📡 Source: Local file

✅ Browser launched (Chrome New Headless + Stealth)

📦 BATCH 1/1 | Hotels 1-5/5

🏨 Hoi An Coco River Resort & Spa | Phòng Loại Sang